<a href="https://colab.research.google.com/github/jdasam/mas1004-assignment1/blob/main/assignment1_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 on Google Colab

Use this if you do not want to install anything on your own computer, or if
your computer is slow.

Colab gives you a machine in a data centre for free. Two things about it will
catch you out, so deal with both before you start:

1. Everything you save on the Colab machine is deleted when your session ends.
   Your downloaded images would go with it. So we keep the whole assignment in
   your Google Drive instead, and Colab just works on it there.
2. It gives you a plain CPU unless you ask for a GPU.

Turn the GPU on. Training ResNet18 on a CPU takes several minutes per run, and
on a GPU it is much faster. The menu at the top, `Runtime` > `Change runtime
type`, pick `T4 GPU`, then `Save`. The session restarts, so do that first.
The free GPU has a daily limit, so turn it back off when you are only looking
at pictures or writing.

## 1. What machine did we get?

In [ ]:
import torch, platform
print("python  ", platform.python_version())
print("torch   ", torch.__version__)
print("GPU     ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, running on CPU")

## 2. Connect your Google Drive

A box will pop up asking you to sign in and allow access. It is asking for your
own Drive, so that the images you download survive after this session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Put the assignment in your Drive

The first time you run this it copies your repository into
`My Drive/mas1004/assignment1`. Every time after that it just moves into the
folder that is already there, so your work is never overwritten.

Replace the address below with your own repository once you have made one.

In [ ]:
import os
from pathlib import Path

STARTER = "https://github.com/jdasam/mas1004-assignment1.git"
HOME = Path("/content/drive/MyDrive/mas1004")
PROJECT = HOME / "assignment1"

HOME.mkdir(parents=True, exist_ok=True)
if not PROJECT.exists():
    !git clone $STARTER "{PROJECT}"
else:
    print("already there, leaving it alone")

os.chdir(PROJECT)
print("working in", Path.cwd())
!ls

## 4. Install what Colab does not already have

PyTorch is already there. This adds the image downloader and the ONNX tools
that put your model on the web page.

In [ ]:
!pip install -q -r requirements.txt

## 5. Check that the given code works

Six tests should pass. These cover code you were given, not code you write.

In [ ]:
!python -m pytest -q tests/test_export.py

## 6. Download your images

Change the search phrases to your own categories. This writes into
`data/raw` inside your Drive, so it is still there tomorrow.

In [ ]:
!python src/collect.py --classes "espresso cup,wine glass,paper coffee cup" --n 150
!cp -r data/raw data/clean

## 7. Look at every image

This draws every image onto sheets. Look at all of them. Do not remove anything
yet: your first training run is on the images as they came.

In [ ]:
!python src/clean.py look
from IPython.display import Image, display
for sheet in sorted(Path("results/cleaning/look").glob("*.png")):
    print(sheet.name)
    display(Image(filename=str(sheet)))

## 8. Put your images from a new source in

Make a folder `data/new_images/<category>` for each category, with exactly the
same names as the folders in `data/clean`, and put at least five images into
each. They must come from a source you can be sure is not in your downloads,
such as photos you took yourself or a friend took, or frames from a video you
recorded. Not another web search. The file browser on the left can upload
them, under `drive/MyDrive/mas1004/assignment1/data`, or you can drop them
into the same folder in Google Drive from your phone.

The second cell checks that none of them is a near copy of a downloaded image.

In [ ]:
for folder in sorted(Path("data/clean").iterdir()):
    mine = Path("data/new_images") / folder.name
    mine.mkdir(parents=True, exist_ok=True)
    print(f"{mine}   {len(list(mine.iterdir()))} images")

In [ ]:
!python src/clean.py overlap
if Path("results/cleaning/overlap.png").exists():
    display(Image(filename="results/cleaning/overlap.png"))

## 9. Work on your functions

Edit `src/data.py`, `src/train.py` and `src/evaluate.py`. You can open them
from the file browser on the left, under `drive/MyDrive/mas1004/assignment1`,
or edit them on your own computer and let Drive sync them across.

Run the tests as you go. The first run downloads the ImageNet weights.

In [ ]:
!python -m pytest -q tests

## 10. Train, before cleaning anything

In [ ]:
!python src/run.py
!python src/run.py --scratch --tag scratch
!python src/run.py --freeze --lr 1e-3 --tag frozen

In [ ]:
for name in ["run_curves.png", "run_confusion.png", "run_worst.png"]:
    if Path(f"results/{name}").exists():
        print(name)
        display(Image(filename=f"results/{name}"))

## 11. Clean your data

Write your rule first. Then ask the rest of your data which images look least
like their class, and which are near copies of each other.

In [ ]:
!python src/clean.py suspects
for sheet in sorted(Path("results/cleaning/suspects").glob("*.png")):
    print(sheet.name)
    display(Image(filename=str(sheet)))

Remove what breaks your rule, with the reason. Repeat this cell as many times
as you need, with your own file names and reasons.

In [ ]:
!python src/clean.py remove wine_glass/0063.jpg wine_glass/0069.jpg --reason "chart, not a photo"

In [ ]:
!python src/clean.py count

## 12. Train on the clean data and compare on your new images

In [ ]:
!python src/run.py --tag clean
!python src/check.py --compare run clean

## 13. Put the clean model on your page and check your work

In [ ]:
!python src/export_web.py --tag clean
!python src/check.py

## 14. Try the demo page without leaving Colab

This serves the `docs/` folder and opens it in a panel. The upload and drawing
tabs work here. The webcam does not, because of how Colab sandboxes the page,
so test the webcam on your own machine or on your published page.

In [ ]:
import subprocess, time
from google.colab import output

server = subprocess.Popen(["python", "-m", "http.server", "-d", "docs", "8000"])
time.sleep(2)
output.serve_kernel_port_as_window(8000)

## 15. Take the finished page away with you

`docs/` is the whole demo, including `model.onnx`, about 45 MB. Download it
and put it into your own repository on your own computer, then commit, push,
and turn on GitHub Pages.

In [ ]:
!zip -qr docs.zip docs
from google.colab import files
files.download("docs.zip")